# Лабораторная работа 6 — Классификация в PySpark ML

In [27]:
import os
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jre1.8.0_471'
os.environ['PATH'] = os.environ['JAVA_HOME'] + r'\bin;' + os.environ.get('PATH', '')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

spark = SparkSession.builder.master('local[*]').appName('lab6').getOrCreate()

---
# Задание 1. Классификация цветков Iris

## Загрузка данных

In [28]:
iris = spark.read.csv('iris.csv', inferSchema=True, header=True)
iris = (iris
    .withColumnRenamed('sepal.length', 'sepal_length')
    .withColumnRenamed('sepal.width',  'sepal_width')
    .withColumnRenamed('petal.length', 'petal_length')
    .withColumnRenamed('petal.width',  'petal_width'))
iris.show(5)
print('Строк:', iris.count())

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|variety|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| Setosa|
|         4.9|        3.0|         1.4|        0.2| Setosa|
|         4.7|        3.2|         1.3|        0.2| Setosa|
|         4.6|        3.1|         1.5|        0.2| Setosa|
|         5.0|        3.6|         1.4|        0.2| Setosa|
+------------+-----------+------------+-----------+-------+
only showing top 5 rows

Строк: 150


## Предобработка признаков

`StringIndexer` превращает строковый класс в числовую метку (0, 1, 2).

`VectorAssembler` собирает несколько колонок в один вектор `features` — именно такой формат принимает LogisticRegression.

In [29]:
label_indexer = StringIndexer(inputCol='variety', outputCol='label')
iris = label_indexer.fit(iris).transform(iris)

assembler = VectorAssembler(
    inputCols=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'],
    outputCol='features'
)
iris = assembler.transform(iris)
iris.select('features', 'label', 'variety').show(5)

+-----------------+-----+-------+
|         features|label|variety|
+-----------------+-----+-------+
|[5.1,3.5,1.4,0.2]|  0.0| Setosa|
|[4.9,3.0,1.4,0.2]|  0.0| Setosa|
|[4.7,3.2,1.3,0.2]|  0.0| Setosa|
|[4.6,3.1,1.5,0.2]|  0.0| Setosa|
|[5.0,3.6,1.4,0.2]|  0.0| Setosa|
+-----------------+-----+-------+
only showing top 5 rows



## Разбивка на train/test и обучение

In [30]:
train, test = iris.randomSplit([0.8, 0.2], seed=42)
print('Train:', train.count(), '| Test:', test.count())

lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=100)
model = lr.fit(train)

predictions = model.transform(test)
predictions.select('variety', 'label', 'prediction', 'probability').show(10)

Train: 126 | Test: 24
+----------+-----+----------+--------------------+
|   variety|label|prediction|         probability|
+----------+-----+----------+--------------------+
|    Setosa|  0.0|       0.0|[1.0,5.8045981223...|
|    Setosa|  0.0|       0.0|[1.0,1.7180006172...|
|    Setosa|  0.0|       0.0|[1.0,1.3749820140...|
|    Setosa|  0.0|       0.0|[1.0,1.6438217837...|
|    Setosa|  0.0|       0.0|[1.0,8.6037985351...|
|Versicolor|  1.0|       1.0|[1.52776446351742...|
|    Setosa|  0.0|       0.0|[1.0,6.1382467728...|
|    Setosa|  0.0|       0.0|[1.0,3.7140206289...|
|    Setosa|  0.0|       0.0|[1.0,1.0992474583...|
|Versicolor|  1.0|       1.0|[6.94841784338202...|
+----------+-----+----------+--------------------+
only showing top 10 rows



## Оценка качества

In [31]:
evaluator = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction')

accuracy  = evaluator.evaluate(predictions, {evaluator.metricName: 'accuracy'})
precision = evaluator.evaluate(predictions, {evaluator.metricName: 'weightedPrecision'})
recall    = evaluator.evaluate(predictions, {evaluator.metricName: 'weightedRecall'})
f1        = evaluator.evaluate(predictions, {evaluator.metricName: 'f1'})

print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1:        {f1:.4f}')

Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1:        1.0000


## Матрица ошибок (Iris)

In [32]:
predictions.groupBy('variety', 'prediction').count().orderBy('variety', 'prediction').show()

+----------+----------+-----+
|   variety|prediction|count|
+----------+----------+-----+
|    Setosa|       0.0|   11|
|Versicolor|       1.0|    6|
| Virginica|       2.0|    7|
+----------+----------+-----+



---
# Задание 2. Классификация пассажиров Титаника


## Загрузка данных

In [33]:
titanic = spark.read.csv('titanic.csv', inferSchema=True, header=True)
titanic.show(5)
print('Строк:', titanic.count())
titanic.printSchema()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

## Предобработка данных

Оставляем только полезные признаки, приводим типы, заполняем пропуски.

In [34]:
mean_age = titanic.select(F.mean('Age')).collect()[0][0]
print(f'Среднее Age: {mean_age:.1f}')

titanic = titanic.select(
    F.col('Survived').cast('double').alias('label'),
    'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'
).fillna({'Age': mean_age, 'Fare': 0.0, 'Embarked': 'S'})

titanic.show(5)
print('Пропуски:')
titanic.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in titanic.columns]
).show()

Среднее Age: 29.7
+-----+------+------+----+-----+-----+-------+--------+
|label|Pclass|   Sex| Age|SibSp|Parch|   Fare|Embarked|
+-----+------+------+----+-----+-----+-------+--------+
|  0.0|     3|  male|22.0|    1|    0|   7.25|       S|
|  1.0|     1|female|38.0|    1|    0|71.2833|       C|
|  1.0|     3|female|26.0|    0|    0|  7.925|       S|
|  1.0|     1|female|35.0|    1|    0|   53.1|       S|
|  0.0|     3|  male|35.0|    0|    0|   8.05|       S|
+-----+------+------+----+-----+-----+-------+--------+
only showing top 5 rows

Пропуски:
+-----+------+---+---+-----+-----+----+--------+
|label|Pclass|Sex|Age|SibSp|Parch|Fare|Embarked|
+-----+------+---+---+-----+-----+----+--------+
|    0|     0|  0|  0|    0|    0|   0|       0|
+-----+------+---+---+-----+-----+----+--------+



## Кодирование категориальных признаков

`StringIndexer` переводит строки в числа (male=0, female=1).

`OneHotEncoder` превращает числа в бинарные векторы — чтобы модель не думала что female «больше» male.

In [35]:
sex_indexer      = StringIndexer(inputCol='Sex',      outputCol='sex_idx')
embarked_indexer = StringIndexer(inputCol='Embarked', outputCol='embarked_idx')

titanic = sex_indexer.fit(titanic).transform(titanic)
titanic = embarked_indexer.fit(titanic).transform(titanic)

encoder = OneHotEncoder(
    inputCols=['sex_idx', 'embarked_idx'],
    outputCols=['sex_ohe', 'embarked_ohe']
)
titanic = encoder.fit(titanic).transform(titanic)

titanic.select('Sex', 'sex_idx', 'sex_ohe', 'Embarked', 'embarked_idx', 'embarked_ohe').show(5)

+------+-------+-------------+--------+------------+-------------+
|   Sex|sex_idx|      sex_ohe|Embarked|embarked_idx| embarked_ohe|
+------+-------+-------------+--------+------------+-------------+
|  male|    0.0|(1,[0],[1.0])|       S|         0.0|(2,[0],[1.0])|
|female|    1.0|    (1,[],[])|       C|         1.0|(2,[1],[1.0])|
|female|    1.0|    (1,[],[])|       S|         0.0|(2,[0],[1.0])|
|female|    1.0|    (1,[],[])|       S|         0.0|(2,[0],[1.0])|
|  male|    0.0|(1,[0],[1.0])|       S|         0.0|(2,[0],[1.0])|
+------+-------+-------------+--------+------------+-------------+
only showing top 5 rows



## Сборка вектора признаков

In [36]:
assembler = VectorAssembler(
    inputCols=['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'sex_ohe', 'embarked_ohe'],
    outputCol='features'
)
titanic = assembler.transform(titanic)
titanic.select('features', 'label').show(5)

+--------------------+-----+
|            features|label|
+--------------------+-----+
|[3.0,22.0,1.0,0.0...|  0.0|
|[1.0,38.0,1.0,0.0...|  1.0|
|(8,[0,1,4,6],[3.0...|  1.0|
|[1.0,35.0,1.0,0.0...|  1.0|
|[3.0,35.0,0.0,0.0...|  0.0|
+--------------------+-----+
only showing top 5 rows



## Разбивка на train/test и обучение

In [37]:
train, test = titanic.randomSplit([0.8, 0.2], seed=42)
print('Train:', train.count(), '| Test:', test.count())

lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=100)
model = lr.fit(train)

predictions = model.transform(test)
predictions.select('label', 'prediction', 'probability').show(10)

Train: 746 | Test: 145
+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|  0.0|       1.0|[0.09067282229230...|
|  0.0|       1.0|[0.41403042009245...|
|  0.0|       1.0|[0.30362609808938...|
|  0.0|       1.0|[0.48931353017850...|
|  0.0|       1.0|[0.49805507196002...|
|  0.0|       1.0|[0.49631603571240...|
|  0.0|       1.0|[0.42239048711625...|
|  0.0|       1.0|[0.43504675335857...|
|  0.0|       0.0|[0.70543462125874...|
|  0.0|       0.0|[0.65466231562900...|
+-----+----------+--------------------+
only showing top 10 rows



## Оценка качества

In [38]:
evaluator = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction')

accuracy  = evaluator.evaluate(predictions, {evaluator.metricName: 'accuracy'})
precision = evaluator.evaluate(predictions, {evaluator.metricName: 'weightedPrecision'})
recall    = evaluator.evaluate(predictions, {evaluator.metricName: 'weightedRecall'})
f1        = evaluator.evaluate(predictions, {evaluator.metricName: 'f1'})

print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1:        {f1:.4f}')

Accuracy:  0.8138
Precision: 0.8131
Recall:    0.8138
F1:        0.8131


## Матрица ошибок (Titanic)

In [39]:
predictions.groupBy('label', 'prediction').count().orderBy('label', 'prediction').show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|   72|
|  0.0|       1.0|   12|
|  1.0|       0.0|   15|
|  1.0|       1.0|   46|
+-----+----------+-----+

